In [1]:
reset -f

In [4]:
import items
from characters import Character, BaseStats

jon = Character('Jon', 'Dwarf', 'Fighter', level=1, bag=items.Bag(), weapon=items.ShortSteelSword())
dan = Character('Dan', 'Elf', 'Wizard', level=2, bag=items.Bag())
bob = Character('Bob', 'Human', 'Hunter', level=3, bag=items.Bag())
jon.bag.add(items.Potion.of('BH'))
bob.bag.add(items.Potion.of('BH'))

In [5]:
for _ in range(3):
    jon.attack(bob, 'normal')

Jon is attacking Bob
Jon hits Bob with an 8-strength.
Bob has now 17 life and lose 8 resistance points.
Jon is attacking Bob
Jon hits Bob with an 16-strength.
Bob has now 1 life and lose 17 resistance points.
Jon is attacking Bob
Bob dodged Jon attack


In [6]:
jon.location

[1, 0, 0]

# Map

In [1]:
import random
import maps
import characters


class Map:
    """World map."""
    def __init__(self, rmap):
        self.rmap = rmap
        self._lmap = self._to_list()
        self.moutain = ['M', '^', 'Ʌ']
        self.forest = ['F', 'ƒ', ' ']
        self.water = ['W', '~']
        self.village = ['V']
    
    @property
    def shape(self):
        """Returns the shape of the map."""
        split = maps.raw_map.split('\n')
        return len(split), max([len(i) for i in split])
    
    def _to_list(self):
        """Returns a list of list representing the map."""
        lmap = []
        for l in self.rmap.split('\n'):
            l = f'{l:<{self.shape[1]}}'
            lmap.append([c for c in l])
        return lmap
    
    def __str__(self):
        return self.rmap
    
    def event(self, location, lvl=1):
        """Triggers an event depending on location and character level."""
        _, l, c = location
        if self._lmap[l][c] in self.moutain:
            number, = random.choices((0, 1, 2, 3), (60, 30, 9, 1))
            return [characters.get_random_char('Mountain', lvl=lvl) for c in range(number)]
        if self._lmap[l][c] in self.forest:
            number, = random.choices((0, 1, 2, 3), (60, 30, 9, 1))
            return [characters.get_random_char('Forest', lvl=lvl) for c in range(number)]

In [30]:
m = Map(maps.raw_map)
event = m.event([0, 2, 2])

for c in event:
    print(c)

Mountain Orc [Orc Hunter Lvl. 1]
	stg: 14         	res: 13 [0]     
	dxt: 21         	spd: 18         
	spi: 11         	chc: 14         
	hp: 14          	mp: 15          
Weapon: Bare Hands
No Bag
Mountain Troll [Troll Wizard Lvl. 1]
	stg: 8          	res: 14 [0]     
	dxt: 15         	spd: 14         
	spi: 15         	chc: 15         
	hp: 15          	mp: 16          
Weapon: Short Staff
No Bag


# Team and confrontation

In [3]:
import random
import characters

team1 = [characters.get_random_char('Mountain', lvl=1, bot=False) for c in range(2)]
team2 = [characters.get_random_char('Mountain', lvl=1) for c in range(3)]


class Team:
    
    def __init__(self, lst):
        self.lst = lst
        
    def __str__(self):
        disp = ""
        for char in self.lst:
            disp += char.__str__() + 2*'\n'
        return disp
    
    def __add__(self, other):
        if isinstance(other, Team):
            return self.lst + other.lst
        raise TypeError('Can only concatenate Team with Team')
    
    def __iter__(self):
        return iter(self.lst)
    
    def __len__(self):
        return len(self.lst)
    
    def __contains__(self, item):
        return item in self.lst
    
    def __getitem__(self, key):
        return self.lst[key]

    
    
class Combat:
    
    def __init__(self, team1, team2):
        self.team1 = team1
        self.team2 = team2
        self.chars = team1 + team2
        self._order_chars()
        self.actions = ['attack', 'spell', 'steal', 'escape']
        
    def _order_chars(self):
        """Order characters following their speed."""
        self.chars.sort(key=lambda char: char.spd, reverse=True)
    
    def start(self):
        """Start combat."""
        while self._check_alive:
            for char in self.chars:
                if char.alive:
                    self.action(char)
                if not char.alive:
                    self.chars.remove(char)

    def action(self, char):
        """To implement in Team, not in Character!"""
        
        other_team = self._other_team(char)
        
        if char.bot:
            self._ai_action(char, other_team)
        else:
            self._pl_action(char, other_team)

    def _ai_action(self, char, other_team):
        
        target = random.choice(other_team)         # Implement algo : Choose as function of level/life/power of char
        char.attack(target, subtype='normal')
        
    def _pl_action(self, char, other_team):
        for idx, action in enumerate(self.actions):
            print(f'{idx+1} -- {action}')
        
        action = int(input('action : '))
        action = self.actions[action-1]
        
        for idx, subaction in enumerate(char.atk_tree.lst):
            print(f'{idx+1} -- {subaction}')

        subaction = int(input('subaction : '))
        subaction = char.atk_tree.lst[subaction-1]

        for idx, c in enumerate(other_team):
            print(f'{idx+1} -- {c.name} [{c.stats.hp}]')
        
        target = int(input('target : '))
        target = other_team[target-1]
        
        getattr(char, action)(target, subtype=subaction)

    @property
    def _check_alive(self):
        """Check if at least one character remains in each team."""
        c1 = any([c in self.team1 for c in self.chars])
        c2 = any([c in self.team2 for c in self.chars])
        return c1 and c2
    
    def _other_team(self, char):
        team = self.team1 if char not in self.team1 else self.team2
        return Team([c for c in team if c.alive])

In [4]:
t1 = Team(team1)
t2 = Team(team2)
c = Combat(t1, t2)
c.start()

1 -- attack
2 -- spell
3 -- steal
4 -- escape
action : 1
1 -- normal
2 -- single
subaction : 2
1 -- Mountain Dwarf [14]
2 -- Mountain Human [14]
3 -- Mountain Dwarf [14]
target : 1
Mountain Troll is attacking Mountain Dwarf
Mountain Dwarf tried to dodge Mountain Troll attack, but failed


AttributeError: 'Attacks' object has no attribute 'single'

In [8]:
for char in t1:
    print(char.alive)
for char in t2:
    print(char.alive)

False
False
True
True
True


In [25]:
l = [1, 2, 3]

In [27]:
l.pop()

2